A proof of concept here is the brochure pipeline pointed at a different document. One model call can gather or clean the source. A second call writes the finished piece from that source only.

## Blueprint

**1. Name the document and the reader.**  
Decide the exact output and who it is for. One document type, one audience. “A one-page product brief for a new sales hire” is a task. “Help the sales team” is not.

**2. Name the source.**  
The model should write from material you supply, not from memory. That source might be a webpage, a spec, a job description, meeting notes, or a customer record. If the source is long, plan to keep only the useful part.

**3. First call: select.**  
Ask the model to pull out only what the document needs: relevant pages, relevant fields, or relevant facts. The result of this call is a short fact list, not the finished document.

**4. Second call: write.**  
Give the model a role, the audience, the format, and the tone. Put the fact list from step 3 in the user prompt. Tell it to use only those facts and to skip anything the source does not say.

**5. Check it by hand.**  
Read one result and ask: did it invent anything, did it miss the main point, and would the real reader use it? Then try a second source of the same kind. Two good runs is enough for a proof of concept.

## Two tasks that fit this well

**New-hire product brief.**  
Source: a product page plus a short internal spec or FAQ. First call extracts what the product does, who it is for, pricing, and limits. Second call writes a one-page brief a new salesperson can read before a call.

**Customer follow-up email.**  
Source: notes from one sales or support call, plus the company’s usual offer. First call extracts the customer’s problem, what was agreed, and any open question. Second call writes a short email in your company’s tone, with a clear next step, and nothing that was not in the notes.

In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents  
from openai import OpenAI


d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [12]:
product_source='https://www.amazon.in/dp/B0H7SCVXYP/?_encoding=UTF8&th=1&pd_rd_w=Oai0v&content-id=amzn1.sym.48adf6f6-bca4-4855-878d-a9da01f53527&pf_rd_p=48adf6f6-bca4-4855-878d-a9da01f53527&pf_rd_r=HFY6N2J2AHSXS79DN2EW&pd_rd_wg=LEg2n&pd_rd_r=1b578220-13b5-4243-83bb-cd3ba33c1fe2&ref_=pd_hp_d_r_atf_unk#'

contents = fetch_website_contents(product_source)
contents

"OnePlus N6x | 4GB+128GB | Ice Blue | 7000mAh Battery | Power for Days | 48-Months Smoothness | Smooth 120Hz Display | Largest 5300mm2 Cooling System | Military-Grade Durability : Amazon.in: Toys & Games\n\nSkip to\nMain content\nAbout this item\nAbout this item\nAbout this item\nBuying options\nCompare with similar items\nVideos\nReviews\nKeyboard shortcuts\nSearch\nalt\n+\n/\nCart\nshift\n+\nalt\n+\nC\nHome\nshift\n+\nalt\n+\nH\nOrders\nshift\n+\nalt\n+\nO\nAdd to cart\nshift\n+\nalt\n+\nK\nProduct summary\nshift\n+\nalt\n+\nD\nShow/Hide shortcuts\nshift\n+\nalt\n+\nZ\nTo move between items, use your keyboard's up or down arrows.\n.in\nDelivering to Dispur 781022\nUpdate location\nAll\nSelect the department you want to search in\nAll Categories\nAlexa Skills\nAmazon Devices\nAmazon Fashion\nAmazon Pharmacy\nAppliances\nApps & Games\nAudible Audiobooks\nBaby\nBeauty\nBooks\nCar & Motorbike\nClothing & Accessories\nCollectibles\nComputers & Accessories\nDeals\nElectronics\nFurniture\nG

In [13]:
content_system_prompt = """
You are provided with the content of a product page found on a webpage.
You are able to extracts what the product does, who it is for, pricing, what the product does not do and limits.

You should respond in JSON as in this example:

{
    "product": [
        {"type": "phone"},
        {"brand": "Nokia"},
        {"price": "$1000"},
        {"who-is-it-for": "for smart users"},
        {"does": "Smart phone features"},
        {"does-not-do": "It cannot fly"}
    ]
}
"""

In [18]:
def get_user_prompt(product_details):
    content_user_prompt = f"""

Please extract the product spoecifications from the product page and
respond details in JSON format.
Here is the content of the product page:
"""
    content_user_prompt += "\n".join(product_details)
    return content_user_prompt

In [20]:
get_user_prompt(contents)

"\n\nPlease extract the product spoecifications from the product page and\nrespond details in JSON format.\nHere is the content of the product page:\nO\nn\ne\nP\nl\nu\ns\n \nN\n6\nx\n \n|\n \n4\nG\nB\n+\n1\n2\n8\nG\nB\n \n|\n \nI\nc\ne\n \nB\nl\nu\ne\n \n|\n \n7\n0\n0\n0\nm\nA\nh\n \nB\na\nt\nt\ne\nr\ny\n \n|\n \nP\no\nw\ne\nr\n \nf\no\nr\n \nD\na\ny\ns\n \n|\n \n4\n8\n-\nM\no\nn\nt\nh\ns\n \nS\nm\no\no\nt\nh\nn\ne\ns\ns\n \n|\n \nS\nm\no\no\nt\nh\n \n1\n2\n0\nH\nz\n \nD\ni\ns\np\nl\na\ny\n \n|\n \nL\na\nr\ng\ne\ns\nt\n \n5\n3\n0\n0\nm\nm\n2\n \nC\no\no\nl\ni\nn\ng\n \nS\ny\ns\nt\ne\nm\n \n|\n \nM\ni\nl\ni\nt\na\nr\ny\n-\nG\nr\na\nd\ne\n \nD\nu\nr\na\nb\ni\nl\ni\nt\ny\n \n:\n \nA\nm\na\nz\no\nn\n.\ni\nn\n:\n \nT\no\ny\ns\n \n&\n \nG\na\nm\ne\ns\n\n\n\n\nS\nk\ni\np\n \nt\no\n\n\nM\na\ni\nn\n \nc\no\nn\nt\ne\nn\nt\n\n\nA\nb\no\nu\nt\n \nt\nh\ni\ns\n \ni\nt\ne\nm\n\n\nA\nb\no\nu\nt\n \nt\nh\ni\ns\n \ni\nt\ne\nm\n\n\nA\nb\no\nu\nt\n \nt\nh\ni\ns\n \ni\nt\ne\nm\n\n\nB\nu\ny\ni\nn\ng\n \no\n

In [21]:
response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": content_system_prompt},
            {"role": "user", "content": get_user_prompt(contents)}
        ],
    )
result = response.choices[0].message.content
display(Markdown(result))

{
  "product": [
    {"type": "smartphone"},
    {"brand": "OnePlus"},
    {"model": "N6x"},
    {"memory": "4GB RAM + 128GB Storage"},
    {"color": "Ice Blue"},
    {"battery": "7000mAh"},
    {"battery_note": "Power for days"},
    {"display": "Smooth 120Hz"},
    {"cooling_system": "Largest 5300mm2 cooling system"},
    {"durability": "Military-grade durability"},
    {"features": "48-months smoothness"},
    {"does": "Provides long battery life and smooth display experience"},
    {"does-not-do": "No specific info about camera, OS version, or network bands"},
    {"who-is-it-for": "Users looking for a durable phone with large battery and smooth display"}
  ]
}